# 05 Merge

**Owner:** Together (one types, one checks every row count)

**What this notebook does:** puts democracy, conflict and arms imports into one table `panel` with one row per country-year.

**Reads:** `vdem.csv`, `ucdp_cy.csv`, `sipri_cy.csv`

**Writes:** `panel.csv`

This is a main notebook. It must run from top to bottom with Kernel → Restart and Run All.
Write simple code in small steps: one operation per line, print the result after each step. Do the merge practice in your sandboxes first.

## Setup

In [69]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

RAW = "../data/raw/"
CLEAN = "../data/clean/"
pd.set_option("display.max_columns", 50)

**Step 1.** Load the three clean files: `vdem`, `ucdp_cy`, `sipri_cy`. Print the shape of each.

In [70]:
# your code here
vdem = pd.read_csv(CLEAN + "vdem.csv")
ucdp_cy = pd.read_csv(CLEAN + "ucdp_cy.csv")
sipri_cy = pd.read_csv(CLEAN + "sipri_cy.csv")

print(vdem.shape)
print(ucdp_cy.shape)
print(sipri_cy.shape)


(12645, 7)
(2043, 5)
(7851, 3)


**Step 2.** Before merging, check two things for all three tables. (a) Each `iso3`-`year` pair appears only once: `duplicated(["iso3", "year"]).sum()` must be 0. (b) Print the columns: apart from `iso3` and `year`, no column name may appear in two tables, otherwise the merge creates `_x` and `_y` columns. If something is wrong, stop and fix the notebook that made the file.

In [71]:
# your code here
print(vdem.duplicated(["iso3", "year"]).sum())
print(ucdp_cy.duplicated(["iso3", "year"]).sum())
print(sipri_cy.duplicated(["iso3", "year"]).sum())

print(vdem.columns)
print(ucdp_cy.columns)
print(sipri_cy.columns)



0
0
0
Index(['country_name', 'iso3', 'cow_code', 'year', 'democracy_index',
       'regime_type', 'regime_label'],
      dtype='str')
Index(['iso3', 'year', 'n_conflicts', 'max_intensity', 'conflict_active'], dtype='str')
Index(['iso3', 'year', 'arms_imports_tiv'], dtype='str')


**Step 3.** Write down the number of rows of `vdem` in a variable `n_start`. V-Dem decides which country-years exist.

In [72]:
# your code here
n_start = len(vdem)


**Step 4.** Left merge `ucdp_cy` onto `vdem` on `["iso3", "year"]` with `validate="one_to_one"` (each country-year once on both sides). Call it `panel`. Print `len(panel)` and compare with `n_start`. They must be equal. (See the week 5 joins lecture and the merge practice in your sandboxes.)

In [73]:
# your code here
panel = pd.merge(vdem, ucdp_cy, on=["iso3", "year"], how="left", validate="one_to_one")
print(len(panel) - n_start)


0


**Step 5.** Which UCDP rows found no partner? Turn it around: left merge the V-Dem keys onto `ucdp_cy`: `check = pd.merge(ucdp_cy, vdem[["iso3", "year"]], on=["iso3", "year"], how="left", validate="one_to_one", indicator=True)`. Rows with `_merge == "left_only"` are conflict years that are missing in V-Dem and get lost in the panel. Print how many there are and look at them. Write in a comment why they are lost (a country-year dropped in 03, or a typo in the crosswalk, which you fix in 01).

In [74]:
# your code here
check = pd.merge(ucdp_cy, vdem[["iso3", "year"]], on=["iso3", "year"], how="left", validate="one_to_one", indicator=True)
print(check.head(5))
missing = check[check["_merge"] == "left_only"]
print(len(missing))
print(missing)

print(vdem[vdem["iso3"] == "BRN"][["iso3", "year"]])
print(vdem[vdem["iso3"] == "CMR"][["iso3", "year"]])
print(vdem[vdem["iso3"] == "GRD"][["iso3", "year"]])

  iso3  year  n_conflicts  max_intensity  conflict_active _merge
0  AFG  1978            1              2                1   both
1  AFG  1979            2              2                1   both
2  AFG  1980            1              2                1   both
3  AFG  1981            1              2                1   both
4  AFG  1982            1              2                1   both
6
    iso3  year  n_conflicts  max_intensity  conflict_active     _merge
193  BRN  1962            1              1                1  left_only
238  CMR  1957            1              1                1  left_only
239  CMR  1958            1              1                1  left_only
240  CMR  1959            1              1                1  left_only
241  CMR  1960            1              2                1  left_only
571  GRD  1983            1              1                1  left_only
Empty DataFrame
Columns: [iso3, year]
Index: []
     iso3  year
2000  CMR  1961
2001  CMR  1962
2002  CMR  1963

In [75]:
vdem_raw = pd.read_csv(RAW + "vdem/V-Dem-CY-Core-v16.csv", usecols=["country_name", "country_text_id", "year", "v2x_regime"])
print(vdem_raw.shape)

brunei = vdem_raw["country_name"].str.contains("Brunei", case=False, na=False)
brunei_year = vdem_raw["year"] == 1962
print(vdem_raw[brunei & brunei_year])

grenada = vdem_raw["country_name"].str.contains("Grenada", case=False, na=False)
grenada_year = vdem_raw["year"] == 1983
print(vdem_raw[grenada & grenada_year])

(28092, 4)
Empty DataFrame
Columns: [country_name, country_text_id, year, v2x_regime]
Index: []
Empty DataFrame
Columns: [country_name, country_text_id, year, v2x_regime]
Index: []



Comment: BRN 1962, CMR 1957-1960, and GRD 1983 are absent from raw V-Dem, so these UCDP rows are not in the panel.

**Step 6.** Country-years without a UCDP row had no conflict. One column per line: fill the missing values in `conflict_active`, `n_conflicts` and `max_intensity` with 0. Print `.isna().sum()` for them. Then, one per line, make them integers with `astype(int)`. (After the merge they are decimals, because missing values force pandas to use floats.)

In [76]:
# your code here
panel["conflict_active"] = panel["conflict_active"].fillna(0)
panel["n_conflicts"] = panel["n_conflicts"].fillna(0)
panel["max_intensity"] = panel["max_intensity"].fillna(0)

print(panel[["conflict_active", "n_conflicts", "max_intensity"]].isna().sum())

panel["conflict_active"] = panel["conflict_active"].astype(int)
panel["n_conflicts"] = panel["n_conflicts"].astype(int)
panel["max_intensity"] = panel["max_intensity"].astype(int)

print(panel.head(10))

conflict_active    0
n_conflicts        0
max_intensity      0
dtype: int64
  country_name iso3  cow_code  year  democracy_index  regime_type  \
0  Afghanistan  AFG     700.0  1950            0.099            0   
1  Afghanistan  AFG     700.0  1951            0.099            0   
2  Afghanistan  AFG     700.0  1952            0.088            0   
3  Afghanistan  AFG     700.0  1953            0.085            0   
4  Afghanistan  AFG     700.0  1954            0.079            0   
5  Afghanistan  AFG     700.0  1955            0.080            0   
6  Afghanistan  AFG     700.0  1956            0.081            0   
7  Afghanistan  AFG     700.0  1957            0.081            0   
8  Afghanistan  AFG     700.0  1958            0.081            0   
9  Afghanistan  AFG     700.0  1959            0.081            0   

       regime_label  n_conflicts  max_intensity  conflict_active  
0  closed autocracy            0              0                0  
1  closed autocracy           

**Step 7.** Left merge `sipri_cy` onto `panel` the same way (`validate="one_to_one"`). Check the row count. Find the lost SIPRI rows with the same `check` merge as in step 5, and write their TIV total in the decisions table. Then fill missing `arms_imports_tiv` with 0.

In [77]:
# your code here
panel = pd.merge(panel, sipri_cy,on=["iso3", "year"], how="left", validate="one_to_one")
print(len(panel))

12645


In [78]:
check = pd.merge(sipri_cy, panel[["iso3", "year"]], on=["iso3", "year"], how="left", validate="one_to_one", indicator=True)
missing = check[check["_merge"] == "left_only"]
print(len(missing))
print(missing)
print(missing["arms_imports_tiv"].sum())

panel["arms_imports_tiv"] = panel["arms_imports_tiv"].fillna(0)
print(panel["arms_imports_tiv"].isna().sum())


67
     iso3  year  arms_imports_tiv     _merge
144   ARE  1968             12.70  left_only
145   ARE  1969             12.30  left_only
146   ARE  1970             66.64  left_only
295   ATG  2022              0.24  left_only
296   ATG  2023              0.30  left_only
...   ...   ...               ...        ...
6575  SSD  2007             37.12  left_only
6576  SSD  2008             44.16  left_only
6577  SSD  2009              1.44  left_only
6578  SSD  2010              1.52  left_only
7470  VCT  1987              3.90  left_only

[67 rows x 4 columns]
1310.88
0


**Step 8.** Spot check three countries by hand. Print the rows for Ukraine 2010 to 2024, Colombia 1985 to 2000, and Switzerland 2015 to 2020. Do the numbers make sense? Write one comment per country.

In [ ]:
# your code here
ukraine_hand = panel[panel["year"].between(2010, 2024) & (panel["iso3"] == "UKR")]
colombia_hand = panel[panel["year"].between(1985, 2000) & (panel["iso3"] == "COL")]
switzerland_hand = panel[panel["year"].between(2015, 2020) & (panel["iso3"] == "CHE")]

print(ukraine_hand)
print(switzerland_hand)
print(colombia_hand)

comment:
ukraine: makes sense, we see the crimean invasion and war in donbass in 2014 and the start of the russian full scale invasion in 2022
colombia: we see the war on drugs and the democracy changing
switzerland: we see democratic stability and decreasing weapon imports

**Step 9.** Save `panel` as `panel.csv` with `index=False`. Commit it, so the app never has to rerun the notebooks.

In [91]:
panel.to_csv(CLEAN + "panel.csv", index=False)


**Checkpoint.** Run this cell. OK means done. Do not change it.

In [92]:
panel_check = pd.read_csv(CLEAN + "panel.csv")
assert len(panel_check) == len(pd.read_csv(CLEAN + "vdem.csv")), "panel must have as many rows as vdem"
assert panel_check.duplicated(["iso3", "year"]).sum() == 0
for col in ["iso3", "year", "country_name", "democracy_index", "regime_type", "regime_label",
            "conflict_active", "n_conflicts", "max_intensity", "arms_imports_tiv"]:
    assert col in panel_check.columns, f"missing {col}"
assert not any(col.endswith("_x") or col.endswith("_y") for col in panel_check.columns), "column name clash in a merge"
for col in ["conflict_active", "n_conflicts", "max_intensity", "arms_imports_tiv"]:
    assert panel_check[col].isna().sum() == 0, f"missing values in {col}"
assert panel_check["conflict_active"].dtype.kind == "i"
assert set(panel_check["conflict_active"].unique()) <= {0, 1}
assert set(panel_check["max_intensity"].unique()) <= {0, 1, 2}
assert panel_check["n_conflicts"].sum() <= pd.read_csv(CLEAN + "ucdp_cy.csv")["n_conflicts"].sum()
print(panel_check.shape)
print("OK")

(12645, 11)
OK


## Decisions in this notebook

Write every choice you made here (and copy it into the decisions log in `CONVENTIONS.md`). You need these for the report.

| Decision | Why |
| --- | --- |
|  |  |

BRN 1962, CMR 1957-1960, and GRD 1983 are absent from raw V-Dem, so these UCDP rows are not in the panel.



67 rows were missing in the panel dataset when we merged sipri_cy on it. Their total TIV sums up to 1310.88